In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def _find_workspace_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "config" / "workspace.json").exists():
            return candidate
    raise FileNotFoundError("Could not locate workspace root from the current working directory.")


WORKSPACE_DIR = _find_workspace_root(Path.cwd())
PAPER_ROOT = WORKSPACE_DIR.parent
RESEARCH_ROOT = PAPER_ROOT.parent
SHARED_ASSETS_DIR = RESEARCH_ROOT / "shared-assets"
SHARED_CODE_DIR = SHARED_ASSETS_DIR / "code"
if str(SHARED_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_CODE_DIR))

from workspace_rooting.workspace_paths import canonical_workspace_paths

paths = canonical_workspace_paths(WORKSPACE_DIR)
WORKSPACE_DIR = paths["workspace"]
CODE_DIR = paths["code"]
CONFIG_DIR = paths["config"]
DATA_DIR = paths["data"]
OUTPUTS_DIR = paths["outputs"]
DOCS_DIR = paths["docs"]
LOCAL_DIR = paths["local"]
SHARED_ASSETS_DIR = paths["shared_assets"]
APPENDIX_DIR = CODE_DIR / "appendix"
APPENDIX_OUTPUTS_DIR = OUTPUTS_DIR / "appendix"
APPENDIX_RUNS_DIR = APPENDIX_OUTPUTS_DIR / "runs"
APPENDIX_FIGURES_DIR = APPENDIX_OUTPUTS_DIR / "figures"
OPENREFINE_DIR = LOCAL_DIR / "curation" / "openrefine"

for candidate in [CODE_DIR, APPENDIX_DIR]:
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

APPENDIX_RUNS_DIR.mkdir(parents=True, exist_ok=True)
APPENDIX_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("workspace_root:", WORKSPACE_DIR)
print("shared_assets_root:", SHARED_ASSETS_DIR)
print("appendix_outputs_root:", APPENDIX_OUTPUTS_DIR)


# Ontology-Clean RDF Triplets

This notebook is the canonical **post-extraction cleaning stage** for the RDF triplet workflow in `semantic-change`.

## Purpose

- load a finished `triplet_flat.parquet` or a manual OpenRefine CSV export
- apply the shared dark-matter normalization and candidate ontology
- inspect what changes before writing anything
- write the cleaned canonical triplet table used by downstream analysis

In [ ]:
from __future__ import annotations

import importlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

import clean_triplets_with_ontology as cto
importlib.reload(cto)


In [ ]:
# Root resolution now comes from the shared workspace bootstrap cell.

## Configuration

Use this notebook in two modes only:

1. clean a finished run output in `runs/<run_id>/triplet_flat.parquet`
2. compare the ontology cleaner against the preserved manual OpenRefine export

The default below assumes you want to clean the retained 10K supplementary run.


In [ ]:
RUN_ID = 'rdf_triplets__gpt_5_mini__supplementary__10k__v1'
MIN_YEAR = 1990
SOURCE_LABEL = 'shared-ontology-cleaner'
PREFER_OPENREFINE = False

# Optional manual override.
# Example:
# INPUT_PATH_OVERRIDE = OPENREFINE_DIR / 'triplet-flat-OR.csv'
INPUT_PATH_OVERRIDE: Path | None = None
OUTPUT_BASE_OVERRIDE: Path | None = None


In [ ]:
input_path, output_base = cto.resolve_paths(
    run_id=RUN_ID,
    input_path=INPUT_PATH_OVERRIDE,
    output_base=OUTPUT_BASE_OVERRIDE,
    prefer_openrefine=PREFER_OPENREFINE,
)

print('input_path:', input_path)
print('output_base:', output_base)
print('shared_module_root:', getattr(cto, 'SHARED_MODULE_ROOT', '(not exposed by module)'))


## Load the raw triplet table


In [ ]:
triplets_raw = cto.read_triplets(input_path)
cto.validate_triplets(triplets_raw)

print('rows:', len(triplets_raw))
print('columns:', triplets_raw.columns.tolist())
display(triplets_raw.head(10))
display(triplets_raw['predicate'].value_counts(dropna=False).rename_axis('predicate').reset_index(name='n'))
display(triplets_raw['year'].describe())


## OpenRefine (optional)


In [ ]:
openrefine_summary = cto.summarize_openrefine_history(OPENREFINE_DIR / 'history.json')
print(json.dumps(openrefine_summary, indent=2))


## Apply cleaning

The cleaner:
- canonicalizes recurring dark-matter candidate terms
- preserves raw `subject` / `object` columns alongside cleaned ones
- downgrades some weak `constitutes` rows to `candidate_for`
- drops obviously weak identity rows


In [ ]:
triplets_cleaned, summary = cto.clean_triplets_frame(
    triplets_raw,
    run_id=RUN_ID,
    min_year=MIN_YEAR,
    source_label=SOURCE_LABEL,
    openrefine_history_path=OPENREFINE_DIR / 'history.json',
    input_path=input_path,
    output_base=output_base,
)

print(json.dumps(summary, indent=2))


## Inspect what changed

These checks are here so the cleaning step remains transparent.


In [ ]:
display(triplets_cleaned.head(10))

display(
    triplets_cleaned['predicate']
    .value_counts(dropna=False)
    .rename_axis('predicate')
    .reset_index(name='n')
)

display(
    triplets_cleaned['subject']
    .value_counts(dropna=False)
    .head(20)
    .rename_axis('subject')
    .reset_index(name='n')
)

display(
    triplets_cleaned['object']
    .value_counts(dropna=False)
    .head(20)
    .rename_axis('object')
    .reset_index(name='n')
)


In [ ]:
changed_rows = triplets_cleaned.loc[
    (triplets_cleaned['subject'] != triplets_cleaned['subject_raw'])
    | (triplets_cleaned['object'] != triplets_cleaned['object_raw'])
    | (triplets_cleaned['predicate'] != triplets_cleaned['predicate_raw'])
].copy()

print('changed rows:', len(changed_rows))

display(
    changed_rows[
        [
            'bibcode', 'year', 'subject_raw', 'subject', 'predicate_raw', 'predicate',
            'object_raw', 'object', 'subject_tags', 'object_tags'
        ]
    ].head(30)
)


## Write canonical cleaned outputs


In [ ]:
cto.write_cleaned_outputs(
    triplets_cleaned,
    summary,
    output_base=output_base,
)

print('wrote:', output_base.with_suffix('.parquet'))
print('wrote:', output_base.with_suffix('.csv'))
print('wrote:', output_base.with_name(output_base.name + '_summary.json'))


## Next step

After this notebook has written the cleaned outputs, use `APDX_3.0.0-build-triplet-assets.R`. It will automatically prefer `triplet_flat_cleaned.parquet` when present.
